# Reasoning Models for Fun and Profit

Since the advent of GPT-3, foundation models have rapidly progressed from single pass transformer models, to multi-step models that can reason over multiple passes. Multi-step reasoning can be applied to more complex problems, where the model benefits from iterative reasoning to arrive at the correct answer.

If you have used [Open AI's o1](https://openai.com/o1/) model, you might have noticed that it "thinks" for longer and goes through a series of steps to arrive at the answer. It does so because it has been trained to produce a ["chain of thought"](https://openai.com/index/learning-to-reason-with-llms/) (CoT) as it reasons through the problem. In the case of o1, OpenAI specifically chose to *hide* the CoT from the user.

o1 shows quite impressive results in several fields, and is capable of answering certain domain questions [as well or better than domain experts](https://venturebeat.com/ai/forget-gpt-5-openai-launches-new-ai-model-family-o1-claiming-phd-level-performance/). In the case of the [MMMU benchmark](https://mmmu-benchmark.github.io/), o1 is as of September 2024 only about 10 points behind the best human performance, and two points ahead of the worst scores for human experts.

The thing is, o1 is a closed model, and we don't know how it reasons besides the little information OpenAI has published. We can't see its chain of thought, and we can't evaluate the intermediate steps it takes when tackling any given problem.

## Enter the Chinese room

The Chinese Room is a thought experiment that was first proposed by John Searle in 1980. The experiment is designed to show that a computer program cannot have a mind, understanding or consciousness, regardless of how intelligently it may behave.

Funny that, as [Deepseek](https://www.deepseek.com), a chinese company, [has released a multi-step model](https://arstechnica.com/ai/2025/01/china-is-catching-up-with-americas-best-reasoning-ai-models/) which discloses its chain of thought, and is entirely open source. It is backed by the chinese [High-Flier](https://www.high-flyer.cn/en/) quantitative hedge fund, and was founded by three alumni from [Zhejiang University](https://en.wikipedia.org/wiki/Zhejiang_University). Zhejiang has amongst its alumni the founders of Alibaba and Tsung-Dao Lee (the 1957 physics Nobel Prize laureate), and is considered one of the top universities in China.

[Deepseek R1](https://api-docs.deepseek.com/news/news250120) is claimed to be on par with OpenAI's o1, and shows some impressive results on multiple benchmarks. R1-Zero, the baseline model on which R1 is based, uses reinforcement learning only, without any further [supervised fine tuning](https://nebius.com/blog/posts/fine-tuning/supervised-fine-tuning). R1 is also much cheaper to run than o1 (\$2.60 per million tokens, vs \$60 per million tokens), which is a big deal for many applications which require scale.

Even the distilled versions of R1 show impressive results, with the 32 billion parameter model beating o1-mini on every single benchmark except for GPQA-Diamond, where it is only three points behind.

```{index} Supervised Fine Tuning
```

```{note}
In Supervised Fine Tuning (SFT), you take a pretrained large language model and directly show it examples of correct or “good” responses in a labeled dataset. This gives the model a clear roadmap for how to behave, so it tends to produce more predictable, consistent answers within the scope of that data.

In contrast, a model trained only with Reinforcement Learning (RL) relies on trial-and-error plus reward signals to figure out the best outputs. There aren’t explicit labeled examples; instead, the model explores various responses and updates its strategy based on which ones earn higher rewards.

A model which produces good results with RL only, without SFT, shows that reasoning capabilities can emerge from the model's architecture and training process, without the need for explicit examples of correct behavior, which is groundbreaking. With RL only, in principle R1 will be able to reason more generally than a model which has been fine-tuned on a specific dataset.
```

There are already [quantized versions of R1](https://huggingface.co/models?other=base_model:quantized:deepseek-ai/DeepSeek-R1-Distill-Qwen-32B) put in the wild by the community, meaning that pretty capable versions of R1 can be run on relatively modest hardware.

## Getting started

### Installing Ollama

You can run distilled versions of the R1 model locally in multiple ways, but the easiest is to use [Ollama](https://ollama.com). Start by downloading the Ollama app, and proceed to then download a version of the model which will fit your hardware (you will likely need 16GB of RAM to run the 8B model, 32GB for 14B model and 64GB+ for the 32B model). There are multiple parameter sizes available, from 1.5B parameters all the way up to 70B parameters. In our case we will use a quantized version of the 32B model.

In my case, my version of Ollama is:

In [2]:
! ollama -v

ollama version is 0.5.7


And I am running it on a 96GB RAM Mac Studio M2 Max.

### Pulling the model

Once Ollama is installed, chose and install an appropriate distilled model. In our case we will use [`unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF`](https://huggingface.co/unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF) quantized to 8 bits, which is a 32 billion parameter model, so reasonably small compared to the original R1 model.

In [3]:
! ollama pull hf.co/unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF:Q8_0

pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling e74588a163eb... 100% ▕████████████████▏  34 GB                         
pulling 8aa4c0321ccd... 100% ▕████████████████▏  493 B                         
pulling c737442a210a... 100% ▕████████████████▏  170 B                         
pulling cd7a10a271fc... 100% ▕████████████████▏  193 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
! ollama list

NAME                                                    ID              SIZE     MODIFIED               
hf.co/unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF:Q8_0    8e12be36faa4    34 GB    Less than a second ago    


## Testing the model

With the model installed, we can run it in preparation for some prompts to make sure it all works.

In [13]:
! ollama run hf.co/unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF:Q8_0 "Who are you?"

⠙ ⠹ ⠹ ⠸ ⠴ ⠴ ⠦ ⠇ ⠇ ⠋ ⠙ ⠙ ⠸ <think>

</think>

Greetings! I'm DeepSeek-R1, an artificial intelligence assistant created by
by DeepSeek. I'm at your service and would be delighted to assist you with 
any inquiries or tasks you may have.



## Integrating with Python

Ollama can be invoked from Python via the `ollama` package, which can be installed in your Python environment via `pip` or `conda`. You then can use it to interact with the model. Let's start with a simple fact based question.

In [ ]:
import textwrap
from ollama import generate

MODEL = 'hf.co/unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF:Q8_0'

def send_message(message:str, temperature:float) -> str:
    """
    Sends a prompt to the model and returns the generated response.
    """
    return generate(model=MODEL, prompt=message, options=dict(temperature=temperature)).response

def format_response(response: str) -> tuple:
    """
    Extracts the chain of thought (CoT) between <think>...</think>
    and the final answer from the response string.
    Then cleans up whitespace and wraps the CoT at 80 characters per line.
    """
    try:
        cot = response.split('<think>')[1].split('</think>')[0]
        answer = response.split('</think>')[1]
    except IndexError:
        # If the tags aren't present, just return the whole response as the answer
        return "", response.strip()

    # Remove extra whitespace from the CoT
    cot = ' '.join(cot.split())
    # Wrap the CoT at 80 characters
    cot = textwrap.fill(cot, width=80)

    return cot, answer.strip()

cot, answer = format_response(
    send_message(message="Why is the sky blue?", temperature=0.6)
)

print(f"Chain of thought:\n\n{cot}")


Chain of thought:

Okay, so I'm trying to figure out why the sky is blue. Hmm, I've heard this
question before, and people usually say it's because of something called
Rayleigh scattering. But wait, what exactly does that mean? Let me break it
down. First, I know that sunlight isn't just one color; it's made up of all the
colors of the rainbow. So when the sun shines, its light is a mix of red,
orange, yellow, green, blue, indigo, and violet. These are different wavelengths
of light, right? Red has longer wavelengths, and blue has shorter ones. Now, as
sunlight passes through Earth's atmosphere, it interacts with the molecules and
tiny particles in the air. I think this interaction causes the light to scatter.
But why does blue get scattered more than other colors? I remember something
about the size of the particles affecting how light scatters. Rayleigh
scattering happens when the particles are much smaller than the wavelength of
light. In the atmosphere, nitrogen and oxygen molecule

## Deeper reasoning

Let us give the model something to chew on which goes beyond just some factual questions. We will give it a simple reasoning problem which involves geography, and see how it performs.

In [7]:
cot, answer = format_response(
    send_message(
        message = """
                    Can an athlete swim from London to Southampton? Not-widthstanding environmental factors or risks,
                    plan how one could do it.
                """,
        temperature=0.6
    )
)

print(f"Chain of thought:\n\n{cot}")

Chain of thought:

Okay, so I need to figure out if an athlete can swim from London to Southampton
without considering environmental factors or risks, and then plan how they might
do it. Hmm, let's break this down step by step. First, I should probably look at
the geographical distance between these two cities. London is in the southeast
of England, and Southampton is on the southern coast. So, they're both in the UK
but not too far apart. I think the straight-line distance isn't that long, maybe
a couple hundred miles? But swimming along the coastline would be longer because
you have to follow the shore. I remember seeing maps where the English Channel
is between England and France, so if someone swims from London towards
Southampton, they might have to go around the Isle of Wight. Wait, where exactly
are these places located? I think London is on the River Thames, which empties
into the North Sea, but to get to Southampton, you'd have to head south along
the coast. So, maybe the athl

The model correctly inferred that there is no direct water route between London and Southampton, and that a swim would be very challenging. It also inferred options between swimming near the coast, or offshore. So it seems to have a basic understanding of the geography, the challenges involved in what was asked, and how to mitigate them.

## Enter maths

Let us now give the model a simple maths problem, and see how it performs. In this case we aren't asking a question which only requires simple formulation, but instead one which requires more deliberate thinking.

In [8]:
cot, answer = format_response(
    send_message(
        message = """
                    Is there a sequence of one or more arithmetic operations which can
                    be applied to each prime number between 10 and 20, for which it would always
                    result in the same value ?
                """,
        temperature=0.5
    )
)

print(f"Chain of thought:\n\n{cot}")

Chain of thought:

First, I need to identify all the prime numbers between 10 and 20. These primes
are 11, 13, 17, and 19. Next, I'll explore possible arithmetic operations that
can be applied consistently to each of these primes to achieve the same result.
The operations could include addition, subtraction, multiplication, division, or
any combination of these. One approach is to consider simple linear
transformations. For instance, if I subtract 10 from each prime: - 11 - 10 = 1 -
13 - 10 = 3 - 17 - 10 = 7 - 19 - 10 = 9 This results in different values, so
subtracting 10 alone doesn't work. Another idea is to look for a common factor
or multiple. However, since all primes are distinct and have no common factors
other than 1, finding a common divisor isn't straightforward. I could also
consider more complex operations, such as raising each prime to a power or
applying modular arithmetic. For example: - 11 mod 5 = 1 - 13 mod 5 = 3 - 17 mod
5 = 2 - 19 mod 5 = 4 Again, this doesn't yield

The reasoning sequence in the chain of thought is interesting. The model identifies the primes and explores a few linear and combined operations. It then works out that multiplying by zero will solve the problem, or so would exponentiation by zero. It correctly works out that this is unlikely to be what is intended by the question. Where it failed was in realising that applying the modulo operation by 2 could be a possible solution, and that this could be a possible answer (albeight a trivial one just like multiplying by zero). It then concludes correctly that it is unlikely that there is a sequence of operations which fully solves the problem.

This isn't exactly a groundbreaking result, but it shows that the model can reason through a simple maths problem, and that it can identify the possible solutions and the limitations of the problem presented with detailed reasoning. Not bad for a distilled 32 billion parameter model!

It finally presents us with the answer.

In [12]:
print(f"Answer:\n\n{answer}")

Answer:

To determine whether there exists a sequence of one or more arithmetic operations that can be applied to each prime number between 10 and 20 (which are **11**, **13**, **17**, and **19**) such that they all result in the same value, let's explore possible operations.

### Step-by-Step Analysis:

1. **List of Primes Between 10 and 20:**
   - **11**, **13**, **17**, **19**

2. **Possible Arithmetic Operations:**
   - Addition (+)
   - Subtraction (-)
   - Multiplication (×)
   - Division (÷)
   - Exponentiation (^)
   - Modulo (%)

3. **Exploring Simple Linear Operations:**
   
   Let's consider applying a simple linear operation to each prime number.

   - **Operation:** Subtract 10 from each prime.
     \[
     \begin{align*}
     11 - 10 &= 1 \\
     13 - 10 &= 3 \\
     17 - 10 &= 7 \\
     19 - 10 &= 9 \\
     \end{align*}
     \]
     *Result:* Different values (1, 3, 7, 9).

   - **Operation:** Subtract 12 from each prime.
     \[
     \begin{align*}
     11 - 12 &= -1 \\

## What's next?

It remains to be seen how chain of thought models operate at scale with real world applications, or how cost effective they will be. However it is pretty clear that the race is on, and that OpenAI and its o1 and o3 models isn't the only game in town. The fact that Deepseek has released an open source model which is on par with o1 is a big deal, especially since this is a model originating in China, and that it is much cheaper to run than o1.

This is particularly important as it shows how quickly and furiously competitors can emerge, plus how fast China is catching up.

In the meantime, in Europe, the only foundation model which even comes close is [Mistral Large 2](https://artificialanalysis.ai/models/mistral-large-2). But at least Europe has an [Act](https://artificialintelligenceact.eu)!

In [10]:
! ollama stop hf.co/unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF:Q8_0